In [4]:
# Define the query
query = """
SELECT *
FROM [dbo].[tbl_errordetails_datamodel]
where file_id > ISNULL((SELECT max(FILE_ID)
        FROM [dbo].[tbl_error_completeness_datamodel]), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Convert result to Spark DataFrame
if rows:
    df = spark.createDataFrame(rows, schema=columns)

df.createOrReplaceTempView('vw_tbl_errordetails_datamodel')

In [8]:
def sp_cc_error_calculation():
    tbl_error_completeness_datamodel= spark.sql(""" SELECT 
                    file_id,
                    value as value,
                    attribute_group AS Attribute_Group,
                    error_id as error_id, 
                    business_attribute_name AS business_attribute_name,
                    dimension as dimension,
                    unique_identifier as  unique_identifier
                    FROM vw_tbl_errordetails_datamodel t
                    WHERE dimension = 'Completeness'
                    ORDER BY file_id, error_id
                    """ )

    tbl_error_conformity_datamodel = spark.sql(""" SELECT 
                    file_id as file_id,
                    value as value,
                    attribute_group AS Attribute_Group,
                    error_id as error_id, 
                    business_attribute_name AS business_attribute_name,
                    dimension as dimension,
                    unique_identifier AS unique_identifier
                    FROM vw_tbl_errordetails_datamodel t
                    WHERE dimension = 'Conformity'
                    ORDER BY file_id, error_id""")

    tbl_error_duplicity_datamodel = spark.sql(""" SELECT 
                    file_id as file_id,
                    value as value,
                    attribute_group AS Attribute_Group,
                    error_id as error_id, 
                    business_attribute_name AS business_attribute_name,
                    dimension as dimension,
                    unique_identifier AS unique_identifier
                    FROM vw_tbl_errordetails_datamodel t
                    WHERE dimension = 'Duplicity'
                    ORDER BY file_id, error_id""")
    
    # tbl_error_completeness_datamodel.createOrReplaceTempView("tbl_error_completeness_datamodel")
    tbl_error_completeness_datamodel.toPandas().to_sql('tbl_error_completeness_datamodel', con=engine, if_exists='append', index=False)
    
    # tbl_error_conformity_datamodel.createOrReplaceTempView("tbl_error_conformity_datamodel")
    tbl_error_conformity_datamodel.toPandas().to_sql('tbl_error_conformity_datamodel', con=engine, if_exists='append', index=False)
    
    # tbl_error_duplicity_datamodel.createOrReplaceTempView("tbl_error_duplicity_datamodel")
    tbl_error_duplicity_datamodel.toPandas().to_sql('tbl_error_duplicity_datamodel', con=engine, if_exists='append', index=False)

In [9]:
sp_cc_error_calculation()